# Chapter 6: Build an index with verifiable provenance

Chapter 6 records measurements as source-bound facts. The live answer tests of this run needed such facts both as supplied context and as reference answers. This notebook resolves every exported fact to its source. The evidence index projects measured values into identifiable records. Every record points back to an exact source field and its file hash. These records supply context and reference answers to the live tests.

**Evidence.** Run 20260923T220909Z repeated the regional study end to end on two machines. `spark-fe12` ran numerical discovery with three new optimizer seeds (74101 to 74103), one live structural proposal and a fresh 200,000-observation confirmation sample. `spark-ef84` ran 70 live answer tests and nine report drafts with `claude-sonnet-5`. The population and frozen logistic predictor are those of the chapter companions; the seeds, proposal and confirmation sample are new, so these numbers should not be combined with the historical run's.

## Setup and source checks

The setup cell locates the data root, the directory whose relative layout the evidence records use. It uses `MODEL_VALIDATION_ROOT` when that variable is set. Otherwise it searches for a repository checkout containing the current directory and then for the copy installed with the `aiv` package. `R` is this run's folder under the data root, and `read` loads a JSON file from it.

`reference.json` was exported from the run's regional results and records their SHA-256 digest. The final assertion recomputes that digest, so every value used below comes from the unchanged results of this run. This notebook reads recorded outputs; rerunning it makes no language-model call.

In [1]:
from pathlib import Path
import json, hashlib, os, platform
import numpy as np
import pandas as pd

# Locate the data root: the directory whose relative layout the evidence records use.
MARKER = "book/notebooks/regional_case/companion.py"

def find_data_root():
    configured = os.environ.get("MODEL_VALIDATION_ROOT")
    if configured:
        root = Path(configured).expanduser().resolve()
        if not (root / MARKER).is_file():
            raise FileNotFoundError(f"MODEL_VALIDATION_ROOT must contain {MARKER}")
        return root, "MODEL_VALIDATION_ROOT"
    for path in (Path.cwd(), *Path.cwd().parents):
        if (path / MARKER).is_file():
            return path, "repository checkout"
    try:
        from aiv.companions import locate
    except ImportError:
        raise FileNotFoundError(
            "Companion data not found. Install the aiv package, open the notebook "
            "inside the repository checkout, or set MODEL_VALIDATION_ROOT.") from None
    return locate()

ROOT, DATA_SOURCE = find_data_root()
LIVE = ROOT / "book/experiments/regional_live"
R = ROOT / "book/experiments/regional_live/runs/20260923T220909Z"
print("Python:", platform.python_version())
print("Data root found from:", DATA_SOURCE)
print("Run:", R.relative_to(ROOT).as_posix())
read=lambda p: json.loads((R/p).read_text(encoding="utf-8"))
results=read('regional/results.json')
reference=read('reference.json')
seeds=reference['seeds']
assert hashlib.sha256((R/'regional/results.json').read_bytes()).hexdigest()==reference['source_sha256']


Python: 3.12.3
Data root found from: repository checkout
Run: book/experiments/regional_live/runs/20260923T220909Z


## Resolve every reference fact

`reference.json` holds 18 facts: the confirmation Brier loss, contrast and share of the reflection and continuation regions at each seed. Each fact records its value, source file, the file's SHA-256 digest and a JSON pointer. `resolve` recomputes the digest, follows the pointer through the source and asserts that the field equals the recorded value. The table lists the facts after all have been checked.

In [2]:
def resolve(record):
 source=R/record['source']
 assert hashlib.sha256(source.read_bytes()).hexdigest()==record['sha256']
 value=json.loads(source.read_text(encoding="utf-8"))
 for part in record['pointer'].strip('/').split('/'):
  value=value[part]
 assert value==record['value']
 return value
for record in reference['facts'].values(): resolve(record)
display(pd.DataFrame(reference['facts']).T)
print('All source pointers and hashes verified')

,value,source,sha256,pointer,unit
region:confirmation/reflection_74101/brier,0.085772,regional/results.json,9c80ebefdb314135d6f900ab0fee3505f3adb1863afd2c...,/confirmation/reflection_74101/brier,squared probability error
region:confirmation/reflection_74101/contrast,0.072514,regional/results.json,9c80ebefdb314135d6f900ab0fee3505f3adb1863afd2c...,/confirmation/reflection_74101/contrast,squared probability error
region:confirmation/reflection_74101/share,0.106785,regional/results.json,9c80ebefdb314135d6f900ab0fee3505f3adb1863afd2c...,/confirmation/reflection_74101/share,fraction
region:confirmation/reflection_74102/brier,0.071811,regional/results.json,9c80ebefdb314135d6f900ab0fee3505f3adb1863afd2c...,/confirmation/reflection_74102/brier,squared probability error
region:confirmation/reflection_74102/contrast,0.058639,regional/results.json,9c80ebefdb314135d6f900ab0fee3505f3adb1863afd2c...,/confirmation/reflection_74102/contrast,squared probability error
region:confirmation/reflection_74102/share,0.13352,regional/results.json,9c80ebefdb314135d6f900ab0fee3505f3adb1863afd2c...,/confirmation/reflection_74102/share,fraction
region:confirmation/reflection_74103/brier,0.084401,regional/results.json,9c80ebefdb314135d6f900ab0fee3505f3adb1863afd2c...,/confirmation/reflection_74103/brier,squared probability error
region:confirmation/reflection_74103/contrast,0.070908,regional/results.json,9c80ebefdb314135d6f900ab0fee3505f3adb1863afd2c...,/confirmation/reflection_74103/contrast,squared probability error
region:confirmation/reflection_74103/share,0.105895,regional/results.json,9c80ebefdb314135d6f900ab0fee3505f3adb1863afd2c...,/confirmation/reflection_74103/share,fraction
region:confirmation/continuation_74101/brier,0.074524,regional/results.json,9c80ebefdb314135d6f900ab0fee3505f3adb1863afd2c...,/confirmation/continuation_74101/brier,squared probability error


All source pointers and hashes verified


**Read the output.** All 18 facts resolve to unchanged fields of `regional/results.json`. The reflection contrasts 0.072514, 0.058639 and 0.070908 and the continuation contrasts 0.05968, 0.059781 and 0.07181 are the values the answer tests expected when source records were supplied. A resolved pointer establishes what the executed artifact records; it does not establish that the synthetic population represents operational AML behavior.